# 1. RS-EoT-4K dataset

In [1]:
# !pip install datasets
# !pip install transformers qwen-vl-utils

In [2]:
import datasets
import random

# Load the dataset from Hugging Face
data = datasets.load_dataset("ShaoRun/RS-EoT-4K")


/home/anhntc/.conda/envs/anhntc_1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Print dataset structure
print(data)


DatasetDict({
    train: Dataset({
        features: ['query', 'response', 'image'],
        num_rows: 4333
    })
})


In [4]:

# Print a random sample
sample = random.choice(data['train'])

for k, v in sample.items():
    print(f'- key: {k}')
    print(f'- val: {v}')
    print('\n\n')

- key: query
- val: Image: <image>
Select the class that best matches the image. Output the letter only (A, B, C, ...).

A. truck
B. gravity_dam
C. soccer_ball_field
D. smoke
E. ground_track_field
F. breakwater



- key: response
- val: Alright, I will reason in a self Q&A style and give the final reply. Start with a global check for major land-cover to narrow down to water-related classes (breakwater, gravity_dam) versus land-based (ground_track_field, soccer_ball_field, truck, smoke). Does a large continuous water body occupy a significant portion of the image? Let’s look at the image, and yes, a very large, dark, continuous area that appears to be a water body occupies the vast majority of the image. It covers almost the entire frame, from the right edge to the bottom, and most of the left and top edges. Having identified a dominant water body, I will check for a coastal protruding structure characteristic of a breakwater to distinguish it from a dam or other classes. Is there a lon

# 2. Model Inference (RS-EoT-7B)

## 2.1. VQA

In [5]:
# !pip install torch torchvision

In [6]:
# !pip install -U transformers accelerate qwen-vl-utils

In [7]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

# Load model and processor
model_name = "ShaoRun/RS-EoT-7B"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)

# Define input image (assumes demo.jpg is in the current directory)
image_path = "./assets/demo.jpg" 

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "How many cars in this image?"},
        ],
    }
]

Loading weights: 100%|██████████| 729/729 [00:05<00:00, 141.57it/s]


In [8]:
# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

In [9]:
# Inference
generated_ids = model.generate(**inputs, max_new_tokens=4096)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print(output_text[0])

Alright, I will reason in a self Q&A style and give the final reply. Start with a global check to confirm whether any cars are present at all before proceeding to regional counting. Are there any cars visible anywhere in the image? Let’s look at the image, yes, there are cars visible in the image. We confirmed cars are present. Next, obtain the total count directly to address the task efficiently. How many cars are visible in the entire image? Let’s look at the image, I can see a total of 5 cars in the entire image. Global observation confirmed the presence of cars. A targeted count across the entire image yielded 5 cars. No contradictions or overlooked regions detected; final checks complete.</think>5


## 2. Visual Grounding with Visualization

In [10]:
import re
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- Helper Functions for Parsing and Visualization ---

def extract_bbox_list_in(text: str) -> list[list[float]]:
    """Extracts bounding boxes from the model output text."""
    boxes = []
    text = re.sub(r'\\([{}\[\]":,])', r'\1', text)
    # Pattern to find lists of numbers like [x1, y1, x2, y2]
    pattern = re.compile(r'\[\s*(.*?)\s*\]', flags=re.IGNORECASE | re.DOTALL)
    matches = pattern.findall(text)
    
    number_pattern = r'-?\d+\.\d+|-?\d+'
    for match in matches:
        nums = re.findall(number_pattern, match)
        if len(nums) >= 4:
            # Take the first 4 numbers as the box
            box = [float(num) for num in nums[:4]]
            boxes.append(box)
    return boxes

def visualize_bboxes(img: Image.Image, boxes: list[list[float]], color=(0, 255, 0), width=3) -> Image.Image:
    """Draws bounding boxes on the image."""
    out = img.copy()
    draw = ImageDraw.Draw(out)
    W, H = img.size
    
    for b in boxes:
        if len(b) < 4: continue
        x1, y1, x2, y2 = b[:4]
        
        # Ensure coordinates are within bounds
        x1, y1 = max(0, min(W-1, x1)), max(0, min(H-1, y1))
        x2, y2 = max(0, min(W-1, x2)), max(0, min(H-1, y2))
        
        # Draw rectangle with thickness
        draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
        
    return out

In [11]:
# --- Main Inference Code ---

model_name = "ShaoRun/RS-EoT-7B"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)

# Load Image
image_path = "./assets/demo.jpg"
image = Image.open(image_path).convert('RGB')

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": 'Locate the black car parked on the right in the remote sensing image. Return the coordinates as "[x1, y1, x2, y2]".'},
        ],
    }
]

# Process Inputs
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Generate
generated_ids = model.generate(**inputs, max_new_tokens=4096)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
response = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Model Response:\n{response}")

# Parse and Visualize
# Extract reasoning and answer. Note: RS-EoT-7B uses <think> tags.
answer_part = response.split("</think>")[-1] 
detection = extract_bbox_list_in(answer_part)

if detection:
    print(f"Detected BBoxes: {detection}")
    vis_img = visualize_bboxes(image, detection)
    vis_img.save("./res.jpg")
    print("Visualization saved to ./res.jpg")
else:
    print("No bounding boxes detected in the response.")

Loading weights: 100%|██████████| 729/729 [00:04<00:00, 160.77it/s]


Model Response:
Alright, I will reason in a self Q&A style and give the final reply. Start with a global check to confirm the presence of a target before localizing it. Is there a black car visible in the image? Let’s look at the image. Yes, there is a black car visible in the image. It is located on the left side of the road, in the upper middle portion of the image. We have confirmed a black car exists and its general location on the left side of the road. Next, we should confirm its vertical position to narrow down the region further. Is the black car positioned in the upper half of the image? Let’s look at the image. Yes, the black car on the left side of the road is positioned in the upper half of the image. We have identified a black car in the upper-left region. To ensure uniqueness before giving the coordinates, we should check if another black car exists in the image. Is there another black car visible in the image besides the one in the upper-left? Let’s look at the image. Ye